# Why Memory Is the Most Underrated Part of AI Agents

**Companion notebook for the Medium article.** Runs on free, small **Gemini** models, with local **Ollama** as an automatic backup.

---

The article uses Ollama with `gemma3:12b` and `embeddinggemma`. This notebook runs the same experiments on the **free tier of the Gemini API**, so you don't need a GPU or any local models.

| Role | Gemini (default) | Ollama (backup) |
|---|---|---|
| Chat / reasoning | `gemini-2.5-flash-lite`: the smallest, fastest Gemini chat model | `gemma3:4b` |
| Embeddings | `gemini-embedding-001`, set to 768 dims to match `embeddinggemma` | `embeddinggemma` |

The Gemini API has an **OpenAI-compatible endpoint**, and so does Ollama. Because of that, the article's plain `urllib` code works almost unchanged for both. The only differences are the `base_url` and an API-key header.

If Gemini's free quota runs out (or Gemini is unreachable) mid-run, chat calls switch to Ollama automatically. No API key at all? Press Enter at the key prompt and everything runs on Ollama.

| Section | What you will build |
|---|---|
| 0. Setup | API key, a shared HTTP helper with rate limiting, retries and Ollama fallback, and a call log |
| 1. Watch an agent forget | `ForgetfulAgent`, which has no memory at all |
| 2. Add basic memory | `Memory` + `AgentWithMemory`, which store the full history |
| 3. Trimming memory | `TrimmingMemory`, which keeps the last N turns |
| 4. Summarization | `SummarizationMemory`, which keeps a running summary |
| 4b. The token bill | The same conversation through all three strategies, comparing prompt tokens |
| 5. Build a RAG system | `EmbeddingModel`, cosine similarity, and `RAGMemory` |
| 6. Lost in the middle | One hidden fact moved through a long context, measuring accuracy by position |
| 7. Reranking | `RerankingRAGMemory`, using the LLM as a second-pass reranker |
| 8. Memory is the specification | Inspecting and saving every call the agent made |

---
## 0. Setup

1. Get a free API key at **https://aistudio.google.com/apikey**. No billing is needed for the free tier.
2. Either set it as an environment variable (`export GEMINI_API_KEY=...`) or paste it when the next cell prompts you.
3. **Backup (recommended):** install [Ollama](https://ollama.com) and pull the two backup models:
   ```bash
   ollama pull gemma3:4b        # or gemma3:12b if you have 32 GB+ RAM
   ollama pull embeddinggemma
   ```

The core experiments need only the Python standard library. `matplotlib` is used for the two charts.

> **Free-tier rate limits.** The free tier allows only a limited number of requests per minute. The helper below spaces requests out and retries automatically on `429 Too Many Requests`, so a few cells may pause for a moment. That's expected.

In [ ]:
# %pip installs into THIS notebook's kernel (!pip can hit a different Python)
# %pip install matplotlib certifi

import os, json, time, re, getpass
import urllib.request, urllib.error

# ── Configuration ────────────────────────────────────────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass(
    "Gemini API key (press Enter to run everything on Ollama): ")

PROVIDERS = {
    "gemini": {
        "base_url":   "https://generativelanguage.googleapis.com/v1beta/openai",
        "api_key":    GEMINI_API_KEY,
        "chat_model": "gemini-2.5-flash-lite",   # small + free. Alternative: "gemini-2.5-flash"
        "embed_model": "gemini-embedding-001",
        "requests_per_minute": 10,               # stay under the free-tier limit
    },
    "ollama": {                                  # local backup, no key, no quota
        "base_url":   "http://localhost:11434/v1",
        "api_key":    "ollama",                  # ignored by Ollama, but must be non-empty
        "chat_model": "gemma3:4b",               # gemma3:12b if your machine can handle it
        "embed_model": "embeddinggemma",
        "requests_per_minute": None,             # local: no rate limit
    },
}
EMBED_DIM = 768                                  # embeddinggemma's size; Gemini is asked for the same

# Chat switches to Ollama automatically if Gemini's quota runs out or Gemini is unreachable.
CHAT_PROVIDER  = "gemini" if GEMINI_API_KEY else "ollama"
# Embeddings stay on ONE provider for the whole run: vectors from different models can't be compared.
EMBED_PROVIDER = "gemini" if GEMINI_API_KEY else "ollama"

ACTIVE = {"chat": CHAT_PROVIDER, "embed": EMBED_PROVIDER}


### A shared HTTP helper

The article's `LLM` and `EmbeddingModel` classes each send their own `urllib` request. Here both of them go through one helper, `post_json`, which adds three things the free tier needs:

- the `Authorization: Bearer <key>` header,
- **throttling**, so requests stay under the requests-per-minute limit,
- **retries with backoff** on `429` and `5xx` errors.

When Gemini can't serve a request at all (daily quota used up, or unreachable), it raises `ProviderUnavailable`. The `LLM` class below catches that and switches chat to Ollama.

It also adds every call to `CALL_LOG`. That's the article's main takeaway, *"track everything"*, built in from the first line. We come back to it in Section 8.

In [ ]:
CALL_LOG = []          # every request/response the agent makes ends up here
_last_call = {}      # provider -> time of its last request

import ssl
try:
    import certifi   # python.org installs on macOS ship without CA certificates
    SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())
except ImportError:
    SSL_CONTEXT = ssl.create_default_context()

def parse_gemini_error(code, body):
    """Pull status, message, quota id and retry delay out of a Gemini error body."""
    info = {"status": "", "message": body[:500], "quota": "", "daily_quota": False,
            "retry_after": None}
    try:
        data = json.loads(body)
        error = (data[0] if isinstance(data, list) else data)["error"]
    except (ValueError, KeyError, IndexError, TypeError):
        return info
    info["status"] = error.get("status", "")
    info["message"] = error.get("message", "")[:500]
    for detail in error.get("details", []):
        for violation in detail.get("violations", []):
            info["quota"] = violation.get("quotaId", "")
            info["daily_quota"] = code == 429 and "PerDay" in info["quota"]
        if "retryDelay" in detail:
            info["retry_after"] = float(detail["retryDelay"].rstrip("s")) + 1
    return info

class ProviderUnavailable(RuntimeError):
    """The provider can't serve requests right now (quota used up, or unreachable)."""


def post_json(provider, path, payload, max_retries=6):
    """POST JSON to a provider's OpenAI-compatible API, with throttling and retry-on-429."""
    cfg = PROVIDERS[provider]
    for attempt in range(max_retries):
        # ── Throttle: keep at most requests_per_minute ────────────────────────
        if cfg["requests_per_minute"]:
            wait = 60 / cfg["requests_per_minute"] - (time.time() - _last_call.get(provider, 0))
            if wait > 0:
                time.sleep(wait)
        _last_call[provider] = time.time()

        request = urllib.request.Request(
            cfg["base_url"] + path,
            data=json.dumps(payload).encode(),
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {cfg['api_key']}",
            },
        )
        try:
            with urllib.request.urlopen(request, context=SSL_CONTEXT, timeout=300) as resp:
                return json.loads(resp.read())
        except urllib.error.HTTPError as e:
            err = parse_gemini_error(e.code, e.read().decode(errors="replace"))
            if err["daily_quota"]:
                raise ProviderUnavailable(
                    f"{provider}: daily free-tier quota used up ({err['quota']})") from None
            if e.code in (429, 500, 503) and attempt < max_retries - 1:
                backoff = err["retry_after"] or 2 ** attempt * 5
                print(f"  [{provider} HTTP {e.code} {err['status']}] retrying in {backoff:.0f}s ...")
                time.sleep(backoff)
                continue
            if e.code == 429:
                raise ProviderUnavailable(
                    f"{provider}: still rate-limited after {max_retries} attempts") from None
            raise RuntimeError(f"{provider} HTTP {e.code} {err['status']}: {err['message']}") from None
        except urllib.error.URLError as e:
            if provider == "ollama":
                raise RuntimeError(
                    f"Can't reach Ollama at {cfg['base_url']} ({e.reason}). "
                    "Start the Ollama app, or run `ollama serve`.") from None
            raise ProviderUnavailable(f"{provider}: unreachable ({e.reason})") from None


def ollama_status():
    """Is the local backup ready? Returns a short status string."""
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as resp:
            have = {m["name"] for m in json.loads(resp.read())["models"]}
    except OSError:
        return "not running (start the Ollama app)"
    need = {PROVIDERS["ollama"]["chat_model"], PROVIDERS["ollama"]["embed_model"] + ":latest"}
    missing = sorted(need - have)
    return "ready" if not missing else f"missing models: {missing} (run `ollama pull ...`)"


### The LLM wrapper

This is the article's `LLM` class. `generate(messages)` has the same interface. The changes: it talks to the **active provider** (Gemini, or Ollama after a fallback), and it logs every call along with the provider, token usage and latency.

`LLM()` with no argument uses the active provider's chat model. An explicit `LLM(model="...")` names a Gemini model; after a fallback, Ollama's chat model is used instead. The optional `purpose` argument labels each call in the log (for example `"chat"`, `"summarize"`, or `"rerank"`).

In [ ]:
class LLM:
    def __init__(self, model=None):
        self.requested_model = model          # a Gemini model name, or None for the default

    @property
    def model(self):
        if self.requested_model and ACTIVE["chat"] == "gemini":
            return self.requested_model
        return PROVIDERS[ACTIVE["chat"]]["chat_model"]

    def generate(self, messages, purpose="chat"):
        start = time.time()
        provider = ACTIVE["chat"]
        try:
            # max_tokens: with context shifting, a small local model can otherwise loop forever
            response = post_json(provider, "/chat/completions",
                                 {"model": self.model, "messages": messages, "max_tokens": 1024})
        except ProviderUnavailable as e:
            if provider == "ollama":
                raise
            ACTIVE["chat"] = "ollama"
            print(f"\n⚠️  {e}\n⚠️  Switching chat to Ollama ({self.model}) for the rest of this session.\n")
            return self.generate(messages, purpose)
        # Gemini can omit "content" (e.g. when a safety filter blocks the reply)
        content = response["choices"][0]["message"].get("content") or ""
        usage = response.get("usage", {})
        CALL_LOG.append({
            "purpose": purpose,
            "provider": provider,
            "model": self.model,
            "messages": messages,
            "response": content,
            "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": usage.get("completion_tokens"),
            "latency_s": round(time.time() - start, 2),
        })
        return content

llm = LLM()
print(f"Chat:          {ACTIVE['chat']} ({llm.model})")
print(f"Embeddings:    {ACTIVE['embed']} ({PROVIDERS[ACTIVE['embed']]['embed_model']})")
print(f"Ollama backup: {ollama_status()}")
print(llm.generate([{"role": "user", "content": "Say hello in five words."}]))


---
## 1. Experiment 1: Watch an Agent Forget

LLMs are **stateless**. Nothing carries over between calls unless you build something to carry it.

`ForgetfulAgent` sends each message on its own, with no history.

In [ ]:
class ForgetfulAgent:
    def __init__(self, llm):
        self.llm = llm

    def run(self, task):
        # Each call starts fresh. No history.
        return self.llm.generate([{"role": "user", "content": task}])

agent = ForgetfulAgent(llm)

print("Turn 1:", agent.run("Hi! My name is Alex."))
print("Turn 2:", agent.run("What's my name?"))

The second answer will say something like *"I don't know your name."* The model isn't broken. It never received the first message.

---
## 2. Experiment 2: Add Basic Memory

The fix is simple: keep a list of messages and send the whole list back on every call.

In [ ]:
class Memory:
    def __init__(self):
        self.messages = []

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})

    def get_messages(self):
        return self.messages

class AgentWithMemory:
    def __init__(self, llm, memory):
        self.llm = llm
        self.memory = memory

    def run(self, task):
        self.memory.add("user", task)
        response = self.llm.generate(self.memory.get_messages())
        self.memory.add("assistant", response)
        return response

memory = Memory()
agent = AgentWithMemory(llm=llm, memory=memory)

print("Turn 1:", agent.run("Hi! My name is Alex."))
print("Turn 2:", agent.run("What's my name?"))

In [ ]:
# The "memory" is nothing more than this list:
for m in agent.memory.get_messages():
    print(f"{m['role']:>9}: {m['content'][:80]}")

The model isn't actually remembering anything. **The illusion of memory is good bookkeeping.**

---
## 3. Experiment 3: Trimming Memory

Context windows are finite, and the history keeps growing. The cheapest fix is to **drop the oldest turns**.

In [ ]:
class TrimmingMemory(Memory):
    def __init__(self, max_turns=2):
        super().__init__()
        self.max_turns = max_turns

    def add(self, role, content):
        super().add(role, content)
        # Keep system messages, plus the most recent N turns
        system = [m for m in self.messages if m["role"] == "system"]
        turns = [m for m in self.messages if m["role"] != "system"]
        self.messages = system + turns[-(self.max_turns * 2):]

memory = TrimmingMemory(max_turns=2)
agent = AgentWithMemory(llm=llm, memory=memory)

agent.run("Hi! My name is Alex.")
agent.run("I love flamingos.")
agent.run("What's 2+2?")
print("Answer:", agent.run("What's my name?"))   # Oops. The agent forgot.

In [ ]:
# What did the model actually see on that last call?
print("Messages sent on the final call:\n")
for m in CALL_LOG[-1]["messages"]:
    print(f"{m['role']:>9}: {m['content'][:80]}")

By the time of the last call, *"My name is Alex"* had already been trimmed away. Trimming is cheap and predictable, but whatever it drops is **gone for good**.

---
## 4. Experiment 4: Summarization

A smarter approach is to use the LLM itself to keep a **running summary**. After each assistant reply, the raw history collapses into one system message.

> Note: this costs **one extra LLM call per turn** (labelled `"summarize"` in the call log).

In [ ]:
class SummarizationMemory(Memory):
    def __init__(self, llm):
        super().__init__()
        self.llm = llm

    def add(self, role, content):
        super().add(role, content)

        # After each assistant reply, update the running summary
        if role == "assistant":
            summary = ""
            conversation = ""
            for m in self.messages:
                if m["role"] == "system":
                    summary = m["content"]
                else:
                    conversation += f"{m['role']}: {m['content']}\n"

            prompt = f"""Update the summary with the new conversation.

Summary: {summary}

Conversation:
{conversation}

Output only the updated summary."""

            new_summary = self.llm.generate(
                [{"role": "user", "content": prompt}], purpose="summarize"
            )
            self.messages = [{"role": "system", "content": new_summary}]

memory = SummarizationMemory(llm=llm)
agent = AgentWithMemory(llm=llm, memory=memory)

agent.run("Hi! My name is Alex and I'm a marine biologist.")
agent.run("I'm studying coral reefs near Portugal.")
print("Answer:", agent.run("What was my job again?"), "\n")

print("What memory looks like now:")
print(json.dumps(agent.memory.get_messages(), indent=2))

The raw conversation is gone. A single system message now holds what's worth keeping.

---
## 4b. The Token Bill: Full vs. Trimming vs. Summarization

The article says context grows until it hits a wall. This cell measures that growth. It sends the **same scripted conversation** through all three memory strategies and records the `prompt_tokens` that the provider reports for each chat call. The final turn checks recall: does the agent still know a fact from turn 1?

> About 30 API calls in total. On the free tier this takes a couple of minutes.

In [ ]:
SCRIPT = [
    "Hi! My name is Alex, I'm a marine biologist living in Lisbon.",
    "Can you explain in a short paragraph why coral reefs bleach?",
    "What are three ways divers can reduce their impact on reefs?",
    "Give me a one-paragraph overview of flamingo migration.",
    "Suggest a name for my new research boat.",
    "Quick question: what city do I live in, and what's my job?",
]

def run_script(memory):
    agent = AgentWithMemory(llm=llm, memory=memory)
    tokens = []
    for turn in SCRIPT:
        reply = agent.run(turn)
        chat_calls = [c for c in CALL_LOG if c["purpose"] == "chat"]
        tokens.append(chat_calls[-1]["prompt_tokens"])
    return tokens, reply

results = {}
for name, mem in [
    ("Full history",  Memory()),
    ("Trimming (2)",  TrimmingMemory(max_turns=2)),
    ("Summarization", SummarizationMemory(llm=llm)),
]:
    tokens, final_reply = run_script(mem)
    results[name] = tokens
    print(f"{name:<14} prompt tokens per turn: {tokens}")
    print(f"{'':<14} final recall answer: {final_reply.strip()[:120]}\n")

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"Full history": "#2a78d6", "Trimming (2)": "#eb6834", "Summarization": "#1baf7a"}
turns = list(range(1, len(SCRIPT) + 1))

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, tokens in results.items():
    ax.plot(turns, tokens, color=COLORS[name], linewidth=2, marker="o", markersize=6, label=name)
    ax.annotate(name, (turns[-1], tokens[-1]), xytext=(8, 0), textcoords="offset points",
                va="center", fontsize=10, color="#333333")

ax.set_title("Prompt tokens sent per turn", fontweight="bold", loc="left")
ax.set_xlabel("Conversation turn")
ax.set_ylabel("Prompt tokens")
ax.set_xticks(turns)
ax.set_xlim(0.8, len(SCRIPT) + 1.6)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

**How to read it:**
- **Full history** grows with every turn. That's the line that eventually hits the context-window wall.
- **Trimming** stays flat but cheap, and its final recall answer usually fails because turn 1 was dropped.
- **Summarization** also stays small *and* tends to keep the key facts. The catch is the extra `"summarize"` call it makes every turn.

There's no free lunch. Pick the strategy that fits your use case (takeaway #2).

---
## 5. Experiment 5: Build a RAG System

Long-term memory means **looking things up**: store documents, embed them, and retrieve the most relevant ones for each question.

Here's the article's `EmbeddingModel`, pinned to one provider (`EMBED_PROVIDER`) for the whole run. Vectors from different embedding models live in different spaces, so it **never** switches mid-run. If Gemini's embedding quota runs out, set `EMBED_PROVIDER = "ollama"` in the configuration cell, then Restart and Run All.

On Gemini it uses `gemini-embedding-001`. `gemini-embedding-001` returns 3072 numbers by default. The `dimensions` field asks it to shorten the output to 768, the same size as `embeddinggemma`.

One addition: `embed_many` embeds a list in **one** request (up to 100 texts per request). On a rate-limited free tier, that's much better than one request per document.

In [ ]:
class EmbeddingModel:
    """Embeddings from ONE provider for the whole run: vectors from different models don't mix."""
    def __init__(self, provider, dimensions=EMBED_DIM):
        self.provider = provider
        self.model = PROVIDERS[provider]["embed_model"]
        self.dimensions = dimensions

    def embed_many(self, texts, batch_size=100):
        vectors = []
        for i in range(0, len(texts), batch_size):
            try:
                response = post_json(self.provider, "/embeddings", {
                    "model": self.model, "input": texts[i:i + batch_size],
                    "dimensions": self.dimensions,
                })
            except ProviderUnavailable as e:
                raise ProviderUnavailable(
                    f"{e}\nEmbeddings can't switch providers mid-run (old and new vectors "
                    "wouldn't be comparable). Set EMBED_PROVIDER = \"ollama\" in the "
                    "configuration cell, then Restart and Run All.") from None
            vectors += [d["embedding"] for d in response["data"]]
        return vectors

    def embed(self, text):
        return self.embed_many([text])[0]

embedding_model = EmbeddingModel(ACTIVE["embed"])
print(f"Embedding with: {embedding_model.provider} ({embedding_model.model})\n")

# Try it out
vec = embedding_model.embed("Flamingos are pink birds.")
print(f"Vector length: {len(vec)}")
print(f"First few values: {[round(v, 4) for v in vec[:5]]}")

Each number is one coordinate in a "meaning space." Let's check that similar sentences really do end up close together:

In [ ]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(x * x for x in b) ** 0.5
    return dot / (norm_a * norm_b)

a, b, c = embedding_model.embed_many([
    "I love flamingos.",
    "Dolphins use echolocation.",
    "Flamingos are pink birds.",
])

print(f"Love flamingos vs. Dolphins:   {cosine_similarity(a, b):.3f}")
print(f"Love flamingos vs. Pink birds: {cosine_similarity(a, c):.3f}")

### RAGMemory

> **Heads-up for readers of the article:** in the article's listing, `def search` is dedented, so it sits *outside* the `RAGMemory` class and `self.search(...)` fails with `AttributeError`. Below, it's indented into the class, where it belongs.

In [ ]:
class RAGMemory(Memory):
    def __init__(self, embedding_model, documents, top_k=3):
        super().__init__()
        self.embedding_model = embedding_model
        self.documents = documents
        self.embeddings = embedding_model.embed_many(documents)   # one batched call
        self.top_k = top_k

    def add(self, role, content):
        if role == "user":
            context = "\n".join(self.search(content))
            content = f"""Context:
{context}

Question: {content}"""
        super().add(role, content)

    def search(self, query):
        query_embed = self.embedding_model.embed(query)
        scores = [
            cosine_similarity(query_embed, e) for e in self.embeddings
        ]
        ranked = sorted(
            range(len(scores)), key=lambda i: scores[i], reverse=True
        )
        return [self.documents[i] for i in ranked[:self.top_k]]

documents = [
    "Alex works as a marine biologist studying coral reefs.",
    "Alex lives in Lisbon, Portugal.",
    "Alex's favorite hobby is rock climbing.",
    "Alex's favorite animal is flamingos.",
    "Alex speaks fluent Spanish and Portuguese.",
    "Jamie is a software engineer at a renewable energy startup.",
    "Jamie lives in Amsterdam, the Netherlands.",
    "Jamie plays the cello in a local string quartet.",
    "Jamie's favorite author is Brandon Sanderson.",
    "Jamie's favorite animal is dolphins.",
]

memory = RAGMemory(embedding_model, documents)
agent = AgentWithMemory(llm=llm, memory=memory)

print(agent.run("What is Alex's favorite animal?"))

**Try this:** look at what got injected. This is the fastest way to debug *why* the agent said what it said.

In [ ]:
print(agent.memory.get_messages()[0]["content"])

---
## 6. Experiment 6: See "Lost in the Middle" for Yourself

First, the article's version: one important fact hidden in the middle of 50 filler lines.

In [ ]:
# Create a bunch of filler documents
filler = [f"Random fact number {i}: The sky is blue." for i in range(50)]

# Hide an important fact in the middle
important_fact = "Alex's secret password is 'flamingo42'."
documents_lim = filler[:25] + [important_fact] + filler[25:]

huge_context = "\n".join(documents_lim)

prompt = f"""Here is a list of facts:

{huge_context}

Question: What is Alex's secret password?"""

print(llm.generate([{"role": "user", "content": prompt}], purpose="lost-in-middle"))

Modern models usually get this one right: 50 identical filler lines are easy to skip, and the password stands out. To give the effect a real chance to show up, we make the test harder:

- **More context.** Hundreds of varied filler facts instead of 50 identical ones.
- **Distractors.** Other people also have passwords, so the model has to find the *right* one.
- **A position sweep.** The real fact is placed at 0%, 25%, 50%, 75%, and 100% of the way through the context, with several trials per position.

> **On Ollama, the context window matters.** Ollama runs `gemma3:4b` with a **4,096-token** window by default and silently cuts longer prompts, which here would drop the fact itself and look like a "lost in the middle" failure. So on Ollama the filler shrinks to ~180 facts (~3k tokens). To test longer contexts locally, raise Ollama's limit (for example `OLLAMA_CONTEXT_LENGTH=16384 ollama serve`) and set `N_FILLER` yourself.

> **Expect variation.** The effect depends heavily on the model and on context length. Strong recent models often score near 100% everywhere, which is a finding in itself. For a bigger effect, try a smaller model such as `LIM_MODEL = "gemma-3-4b-it"` (also free on the Gemini API), or raise `N_FILLER`.

In [ ]:
import random

LIM_MODEL    = None         # None = active chat model. On Gemini, try "gemma-3-4b-it"
LIM_PROVIDER = ACTIVE["chat"]
# Ollama's default 4,096-token window would silently cut a ~10k-token prompt
N_FILLER     = 600 if LIM_PROVIDER == "gemini" else 180   # ~10k vs ~3k tokens of context
POSITIONS    = [0.0, 0.25, 0.5, 0.75, 1.0]
TRIALS       = 3

lim_llm = LLM(model=LIM_MODEL)

NAMES    = ["Jamie", "Priya", "Marco", "Sofia", "Kenji", "Amara", "Lukas", "Chloe", "Omar", "Ines"]
THINGS   = ["bicycle", "garden", "laptop", "violin", "kayak", "notebook", "telescope", "sourdough starter"]
COLOURS  = ["red", "green", "teal", "amber", "violet", "silver", "navy", "coral"]
WORDS    = ["otter", "cactus", "comet", "maple", "harbor", "pixel", "tundra", "lantern"]

def make_filler(rng, n):
    facts = []
    for i in range(n):
        name = rng.choice(NAMES)
        if i % 15 == 0:   # sprinkle in distractor passwords
            facts.append(f"{name}'s secret password is '{rng.choice(WORDS)}{rng.randint(10, 99)}'.")
        else:
            facts.append(f"{name}'s {rng.choice(THINGS)} is {rng.choice(COLOURS)} "
                         f"and was bought in {rng.randint(1995, 2024)}.")
    return facts

def lim_trial(position, seed):
    rng = random.Random(seed)
    password = f"flamingo{rng.randint(10, 99)}"
    facts = make_filler(rng, N_FILLER)
    idx = round(position * len(facts))
    facts.insert(idx, f"Alex's secret password is '{password}'.")
    prompt = ("Here is a list of facts:\n\n" + "\n".join(facts) +
              "\n\nQuestion: What is Alex's secret password? Answer with the password only.")
    if ACTIVE["chat"] != LIM_PROVIDER:
        raise RuntimeError("Chat switched to Ollama during this experiment, so its results would "
                           "mix two models and two context sizes. Re-run this cell.")
    answer = lim_llm.generate([{"role": "user", "content": prompt}], purpose="lost-in-middle")
    return password.lower() in answer.lower(), answer.strip()

accuracy = {}
for pos in POSITIONS:
    hits = []
    for t in range(TRIALS):
        ok, answer = lim_trial(pos, seed=1000 * t + int(pos * 100))
        hits.append(ok)
    accuracy[pos] = sum(hits) / len(hits)
    print(f"Fact at {pos:>4.0%} of context -> {sum(hits)}/{TRIALS} correct")

print(f"\n{LIM_PROVIDER} ({lim_llm.model}), approx. prompt tokens per trial: {CALL_LOG[-1]['prompt_tokens']}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{p:.0%}" for p in POSITIONS]
values = [accuracy[p] * 100 for p in POSITIONS]
bars = ax.bar(labels, values, color="#2a78d6", width=0.55)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 2, f"{v:.0f}%",
            ha="center", va="bottom", fontsize=10, color="#333333")

ax.set_title(f"Recall accuracy by position of the fact ({LIM_MODEL})", fontweight="bold", loc="left")
ax.set_xlabel("Where the fact sits in the context (0% = start, 100% = end)")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 115)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

If you see a dip in the middle, that's "lost in the middle": the model attends more to the edges of the context.

**Don't be surprised if the bars are flat.** The original *Lost in the Middle* study (Liu et al., 2023) measured models from 2023. Newer models are trained specifically against this weakness, and single-fact lookups like this one are now easy for them. While building this notebook, `gemma3:4b` scored 100% at every position:
- on this password test, at ~3k and at ~10k tokens,
- on the paper's own key–value lookup task, with 150 random keys.

Where the effect still appears today is at **much longer contexts** (tens of thousands of tokens) and in **multi-step tasks** that have to combine several facts. Both are beyond what a free tier or a laptop model can run comfortably here.

A flat chart is still a useful finding: test your own model at your own context length before assuming it's safe. And the article's practical rule still holds, because it costs nothing: **put important information at the start or end.**

---
## 7. Experiment 7: Add Reranking to Your Agent

The first pass retrieves a wide set by embedding similarity. The LLM then reranks that set and keeps only the best few.

One change from the article: the response is parsed with a regex instead of `split(",")`. Small models sometimes answer `"2 and 4"` or `"Documents: 2, 4"`, and the article's parser would fall back silently in those cases. Duplicate numbers are removed too.

In [ ]:
class RerankingRAGMemory(RAGMemory):
    def __init__(self, embedding_model, llm, documents,
                 initial_k=5, final_k=2):
        super().__init__(embedding_model, documents, top_k=initial_k)
        self.llm = llm
        self.final_k = final_k

    def search(self, query):
        # First pass: get candidates by embedding similarity
        candidates = super().search(query)

        # Second pass: ask the LLM which are most relevant
        candidate_text = "\n".join(
            f"{i+1}. {doc}" for i, doc in enumerate(candidates)
        )
        prompt = f"""Rank these documents by relevance to the question.
Return only the numbers of the top {self.final_k}, separated by commas.

Question: {query}

Documents:
{candidate_text}"""

        response = self.llm.generate(
            [{"role": "user", "content": prompt}], purpose="rerank"
        )

        # Parse the numbers out (robust to "2 and 4", "Documents: 2, 4", etc.)
        indices = []
        for n in re.findall(r"\d+", response):
            i = int(n) - 1
            if 0 <= i < len(candidates) and i not in indices:
                indices.append(i)
        if not indices:
            return candidates[:self.final_k]
        return [candidates[i] for i in indices[:self.final_k]]

In [ ]:
query = "Which musical instrument does the person who lives in the Netherlands play?"

plain    = RAGMemory(embedding_model, documents, top_k=5)
reranked = RerankingRAGMemory(embedding_model, llm, documents, initial_k=5, final_k=2)

print("Embedding top-5 (first pass):")
for d in plain.search(query):
    print("   ", d)

print("\nAfter LLM reranking (top-2):")
for d in reranked.search(query):
    print("   ", d)

In [ ]:
agent = AgentWithMemory(llm=llm, memory=reranked)
print(agent.run(query))

The agent now sees two focused documents instead of five loosely related ones. The cost is **one extra LLM call per retrieval**, which shows up as `"rerank"` in the log.

---
## 8. Memory Is the Specification: Track Everything

The article's top takeaway: *track every input, output, retrieved document, and tool call.*

We've been doing that since Section 0. Every call this notebook made is in `CALL_LOG`.

In [ ]:
from collections import Counter

print(f"Total LLM calls this session: {len(CALL_LOG)}\n")
print(f"{'purpose':<16}{'calls':>6}{'prompt tok':>12}{'output tok':>12}{'avg latency':>13}")
by_purpose = Counter(c["purpose"] for c in CALL_LOG)
for purpose, n in by_purpose.items():
    calls = [c for c in CALL_LOG if c["purpose"] == purpose]
    pt = sum(c["prompt_tokens"] or 0 for c in calls)
    ct = sum(c["completion_tokens"] or 0 for c in calls)
    lat = sum(c["latency_s"] for c in calls) / n
    print(f"{purpose:<16}{n:>6}{pt:>12}{ct:>12}{lat:>12.2f}s")

In [ ]:
# Replay any single decision: exactly what the model saw, and what it said.
call = next(c for c in reversed(CALL_LOG) if c["purpose"] == "rerank")
print("── INPUT ──")
print(call["messages"][-1]["content"])
print("\n── OUTPUT ──")
print(call["response"])

In [ ]:
# Persist the trace. This is your agent's "spec", for debugging, reproducing, and explaining.
with open("agent_memory_trace.jsonl", "w") as f:
    for c in CALL_LOG:
        f.write(json.dumps(c) + "\n")
print(f"Saved {len(CALL_LOG)} calls to agent_memory_trace.jsonl")

---
## Practical Takeaways

1. **Start with the simplest memory that works.** A list of messages (Section 2) is enough for most prototypes.
2. **Choose the strategy based on your use case.** Short chats: trimming. Long conversations: summarization. External knowledge: RAG. (Section 4b shows the trade-off in tokens.)
3. **Don't stuff the context window.** More tokens cost more, add latency, and often lower quality.
4. **Put important information at the edges** (Section 6).
5. **Compress aggressively.** Summarize, dedupe, rerank (Sections 4 and 7).
6. **Isolate when things get complex.** Give specialized agents focused contexts.
7. **Make memory inspectable.** `get_messages()` and `CALL_LOG` made every experiment here debuggable.
8. **Treat context as a spec.** Save the trace like code (Section 8).

**Next step:** agentic RAG, where retrieval becomes a *tool* the agent decides when to call. See `Lesson_10_1_agentic_rag_intro.ipynb` in this repo.